# Semana 6 · Retrieval Engineering

**Pregunta central:** ¿qué determina qué información termina viendo el modelo?

La semana 5 dejó un RAG que funciona a veces. Hoy se abre el retriever y se mide cada decisión contra un gold set: cómo se parte el documento, con qué modelo se embebe, cuántos fragmentos se recuperan y qué metadata se guarda. Ninguna configuración se acepta sin un número.

Corre en Colab sin llave de API. Todo el trabajo es retrieval; no hace falta un LLM.

## 0. Preparación

In [ ]:
# En Colab: instala lo que falta. En local con el .venv del curso ya está todo.
!pip install -q sentence-transformers faiss-cpu chromadb pypdf 2>/dev/null || true

In [ ]:
# Llave de OpenAI. En Colab: panel de secretos (icono de llave) con el nombre OPENAI_API_KEY
# y acceso activado para este notebook. En Jupyter local: variable de entorno antes de abrir.
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # fuera de Colab, o sin secreto: se usa la variable de entorno si existe
print("llave cargada" if os.environ.get("OPENAI_API_KEY") else "sin llave: se usará el modelo local")

In [ ]:
import os, re, json, time, warnings
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)

In [ ]:
# Copia de examples/rag_lib.py para que el notebook corra en Colab sin clonar el repositorio.
# La versión canónica de las métricas está en course/shared/retrieval_eval.py.
from __future__ import annotations

import json
import re
import urllib.request
from pathlib import Path

import numpy as np

ARCHIVOS = [
    "01-manual-facturacion-cfdi.md", "02-manual-usuarios-e-invitaciones.md",
    "03-manual-reportes-y-exportacion.md", "04-manual-integraciones-y-api.md",
    "05-preguntas-frecuentes.md", "06-politica-reembolsos-y-cargos.md",
    "07-politica-cancelacion-y-eliminacion.md", "08-terminos-del-servicio-y-sla.md",
    "09-catalogo-codigos-de-error.md", "10-notas-de-version.md",
    "11-guia-de-seguridad.md", "12-comunicado-externo.md",
]
TIPO_DOC = {"01": "manual", "02": "manual", "03": "manual", "04": "manual", "05": "faq",
            "06": "politica", "07": "politica", "08": "terminos", "09": "catalogo",
            "10": "notas", "11": "seguridad", "12": "comunicado"}
RAW = "https://raw.githubusercontent.com/JunBarrs24/itm-maestria-llms/main/shared/datasets/corpus/"


# ------------------------------------------------------------------ carga
def cargar_corpus(local: str | Path | None = None) -> dict[str, str]:
    """Devuelve {archivo: texto}. Usa la carpeta local si existe; si no, descarga del repositorio."""
    docs = {}
    base = Path(local) if local else None
    for nombre in ARCHIVOS:
        if base and (base / nombre).exists():
            docs[nombre] = (base / nombre).read_text(encoding="utf-8")
        else:
            with urllib.request.urlopen(RAW + nombre) as r:
                docs[nombre] = r.read().decode("utf-8")
    return docs


def cargar_gold(local: str | Path | None = None) -> list[dict]:
    base = Path(local) if local else None
    if base and (base / "gold_set.jsonl").exists():
        texto = (base / "gold_set.jsonl").read_text(encoding="utf-8")
    else:
        with urllib.request.urlopen(RAW + "gold_set.jsonl") as r:
            texto = r.read().decode("utf-8")
    return [json.loads(l) for l in texto.splitlines() if l.strip()]


# ------------------------------------------------------------------ estructura
def secciones(texto: str) -> list[dict]:
    """Parte un markdown por encabezados ## y ###. Cada sección conserva su encabezado."""
    out, actual, buf = [], None, []
    for linea in texto.splitlines():
        m = re.match(r"^(#{2,3})\s+(.*)", linea)
        if m:
            if actual is not None:
                out.append({"seccion": actual, "texto": "\n".join(buf).strip()})
            actual, buf = m.group(2).strip(), [linea]
        else:
            buf.append(linea)
    if actual is not None:
        out.append({"seccion": actual, "texto": "\n".join(buf).strip()})
    return out


def seccion_en(texto: str, pos: int) -> str | None:
    """Encabezado ## o ### vigente en la posición pos del texto."""
    ultimo = None
    for m in re.finditer(r"^(#{2,3})\s+(.*)$", texto[:pos + 1], re.M):
        ultimo = m.group(2).strip()
    return ultimo


# ------------------------------------------------------------------ chunking
def chunk_por_seccion(docs: dict[str, str]) -> list[dict]:
    chunks = []
    for archivo, texto in docs.items():
        for s in secciones(texto):
            chunks.append({"archivo": archivo, "seccion": s["seccion"], "tipo": TIPO_DOC[archivo[:2]],
                           "texto": s["texto"]})
    return chunks


def chunk_seccion_contexto(docs: dict[str, str], minimo_palabras: int = 8) -> list[dict]:
    """Como chunk_por_seccion, pero cada chunk lleva al inicio su ruta jerárquica
    (título del documento > sección ## > subsección ###) y se omiten los encabezados
    sin cuerpo propio. El texto extra le da al embedding el contexto que el fragmento perdió."""
    out = []
    for archivo, texto in docs.items():
        m = re.search(r"^# (.+)$", texto, re.M)
        titulo = m.group(1).strip() if m else archivo
        padre = None
        for s in secciones(texto):
            nivel2 = s["texto"].startswith("## ")
            cuerpo = "\n".join(s["texto"].splitlines()[1:]).strip()
            if nivel2:
                padre = s["seccion"]
            if len(cuerpo.split()) < minimo_palabras:
                continue
            ruta = f"{titulo} > {padre} > {s['seccion']}" if (not nivel2 and padre) else f"{titulo} > {s['seccion']}"
            out.append({"archivo": archivo, "seccion": s["seccion"], "tipo": TIPO_DOC[archivo[:2]],
                        "texto": ruta + "\n\n" + cuerpo})
    return out


def chunk_fijo(docs: dict[str, str], tamano: int, overlap: int, contar) -> list[dict]:
    """Ventanas de `tamano` tokens con `overlap` tokens de traslape, medidas en palabras.
    `contar(texto)` devuelve tokens; se usa para calibrar palabras por token en cada documento.
    Un chunk fijo puede cruzar varias secciones: `seccion` es la vigente al inicio y
    `secciones` la lista de todas las que toca."""
    chunks = []
    for archivo, texto in docs.items():
        palabras = [(m.group(0), m.start(), m.end()) for m in re.finditer(r"\S+", texto)]
        if not palabras:
            continue
        encabezados = [(m.start(), m.group(2).strip()) for m in re.finditer(r"^(#{2,3})\s+(.*)$", texto, re.M)]
        ratio = contar(texto) / len(palabras)              # tokens por palabra en este documento
        paso_p = max(1, int(tamano / ratio))                # palabras por chunk
        over_p = int(overlap / ratio)
        i = 0
        while i < len(palabras):
            ventana = palabras[i:i + paso_p]
            ini, fin = ventana[0][1], ventana[-1][2]
            vigente = None
            tocadas = []
            for pos, h in encabezados:
                if pos <= ini:
                    vigente = h
                elif pos < fin:
                    tocadas.append(h)
            secs = ([vigente] if vigente else []) + tocadas
            chunks.append({"archivo": archivo, "seccion": vigente, "secciones": secs,
                           "tipo": TIPO_DOC[archivo[:2]], "texto": texto[ini:fin]})
            if i + paso_p >= len(palabras):
                break
            i += max(1, paso_p - over_p)
    return chunks


def chunk_recursivo(docs: dict[str, str], tamano: int, contar) -> list[dict]:
    """Parte por secciones; si una sección excede `tamano` tokens, la parte por párrafos;
    si un párrafo excede, por oraciones. Nunca corta a media palabra."""
    def partir(texto: str) -> list[str]:
        if contar(texto) <= tamano:
            return [texto]
        partes = [p for p in re.split(r"\n\s*\n", texto) if p.strip()]
        if len(partes) == 1:
            partes = [p for p in re.split(r"(?<=[.!?])\s+", texto) if p.strip()]
            if len(partes) == 1:
                return [texto]
        out, buf = [], ""
        for p in partes:
            cand = (buf + "\n\n" + p).strip() if buf else p
            if contar(cand) <= tamano:
                buf = cand
            else:
                if buf:
                    out.append(buf)
                buf = p
        if buf:
            out.append(buf)
        return out
    chunks = []
    for archivo, texto in docs.items():
        for s in secciones(texto):
            for t in partir(s["texto"]):
                chunks.append({"archivo": archivo, "seccion": s["seccion"], "tipo": TIPO_DOC[archivo[:2]], "texto": t})
    return chunks


# ------------------------------------------------------------------ índice y retrieval
class Indice:
    """Índice en numpy: embeddings normalizados, similitud coseno por producto punto."""

    def __init__(self, modelo, chunks: list[dict], max_seq_length: int | None = None):
        self.modelo, self.chunks = modelo, chunks
        if max_seq_length:
            modelo.max_seq_length = max_seq_length
        self.E = modelo.encode([c["texto"] for c in chunks], normalize_embeddings=True, batch_size=32,
                               show_progress_bar=False)

    def buscar(self, pregunta: str, k: int = 5, filtro=None) -> list[dict]:
        q = self.modelo.encode([pregunta], normalize_embeddings=True, show_progress_bar=False)[0]
        scores = self.E @ q
        orden = np.argsort(-scores)
        out = []
        for i in orden:
            c = self.chunks[i]
            if filtro and not filtro(c):
                continue
            out.append({**c, "score": float(scores[i])})
            if len(out) == k:
                break
        return out


# ------------------------------------------------------------------ métricas (copia de shared/retrieval_eval.py)
def es_relevante(chunk: dict, evidencias: list[dict], por_seccion: bool = True) -> bool:
    for ev in evidencias:
        if chunk.get("archivo") != ev.get("archivo"):
            continue
        if not por_seccion or not ev.get("seccion"):
            return True
        if (chunk.get("seccion") or "") == ev["seccion"]:
            return True
        if ev["seccion"] in (chunk.get("secciones") or []):   # chunk fijo que cruza secciones
            return True
    return False


def metricas_pregunta(chunks: list[dict], evidencias: list[dict], k: int, por_seccion: bool = True) -> dict:
    top = chunks[:k]
    hits = [es_relevante(c, evidencias, por_seccion) for c in top]
    encontradas = set()
    for c in top:
        for ev in evidencias:
            if es_relevante(c, [ev], por_seccion):
                encontradas.add(ev["archivo"] + "#" + (ev.get("seccion") or "" if por_seccion else ""))
    total = len({ev["archivo"] + "#" + (ev.get("seccion") or "" if por_seccion else "") for ev in evidencias})
    mrr = next((1.0 / i for i, h in enumerate(hits, start=1) if h), 0.0)
    return {"hit": float(any(hits)), "recall": len(encontradas) / total if total else 0.0, "mrr": mrr}


def evaluar(gold: list[dict], retriever, k: int = 5, por_seccion: bool = True, tipos: set | None = None) -> dict:
    filas = []
    for g in gold:
        if not g.get("evidencia"):
            continue
        if tipos and g.get("tipo") not in tipos:
            continue
        m = metricas_pregunta(retriever(g["pregunta"]), g["evidencia"], k, por_seccion)
        filas.append({**m, "id": g["id"], "tipo": g["tipo"]})
    n = len(filas)
    return {"k": k, "preguntas": n,
            f"hit@{k}": round(sum(f["hit"] for f in filas) / n, 3) if n else 0.0,
            f"recall@{k}": round(sum(f["recall"] for f in filas) / n, 3) if n else 0.0,
            "mrr": round(sum(f["mrr"] for f in filas) / n, 3) if n else 0.0,
            "por_pregunta": filas}

In [ ]:
# El corpus de Facturio: 12 documentos indexables y un gold set de 30 preguntas.
# Si el notebook corre dentro del repositorio usa la carpeta local; si no, descarga de GitHub.
LOCAL = Path("../../shared/datasets/corpus")
docs = cargar_corpus(LOCAL if LOCAL.exists() else None)
gold = cargar_gold(LOCAL if LOCAL.exists() else None)
print(f"{len(docs)} documentos, {sum(len(t.split()) for t in docs.values())} palabras")
pd.DataFrame(gold)["tipo"].value_counts().rename("preguntas").to_frame().T

In [ ]:
MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
modelo = SentenceTransformer(MODELO)
contar = lambda t: len(modelo.tokenizer.encode(t, add_special_tokens=False))   # tokens según el modelo de embeddings
print("dimensión:", modelo.get_sentence_embedding_dimension(), "| max_seq_length por defecto:", modelo.max_seq_length)

## 1. Línea base y métricas

Antes de tocar nada: el RAG de la semana 5 (chunks por sección, modelo multilingüe) medido contra el gold set con hit@k, recall@k y MRR.

In [ ]:
sec = chunk_por_seccion(docs)
idx_sec = Indice(modelo, sec)
def tabla_k(idx, ks=(1, 3, 5)):
    filas = []
    for k in ks:
        r = evaluar(gold, lambda q: idx.buscar(q, k=10), k=k)
        filas.append({"k": k, "hit@k": r[f"hit@{k}"], "recall@k": r[f"recall@{k}"], "mrr": r["mrr"]})
    return pd.DataFrame(filas).set_index("k")
tabla_k(idx_sec)

**Pregunta 1.** ¿Por qué MRR es menor que hit@5? Escribe en una línea qué mide cada uno.

In [ ]:
# TODO: responde en texto.
RESPUESTA_1 = None

## 2. Barrido de chunk size (actividad 1)

**Predicción antes de ejecutar.** Al pasar de chunks fijos de 200 tokens a 500 y a 1000, ¿hit@3 sube, baja o queda igual? ¿Qué estrategia gastará más tokens de contexto con k = 5?

In [ ]:
# TODO: escribe tus predicciones antes de ejecutar la siguiente celda.
MI_PREDICCION = {
    "hit@3 con chunks de 500 vs 200": None,      # "sube" | "baja" | "igual"
    "hit@3 con chunks de 1000 vs 200": None,
    "estrategia con más tokens de contexto @5": None,   # "fijo 200" | "sección" | "sección+ctx"
    "por qué": None,
}

In [ ]:
modelo.max_seq_length = 512
def fila_barrido(nombre, tam, chunks, idx):
    fila = {"estrategia": nombre, "tamaño": tam, "chunks": len(chunks),
            "tok/chunk": sum(contar(c["texto"]) for c in chunks) // len(chunks)}
    for k in (1, 3, 5):
        fila[f"hit@{k}"] = evaluar(gold, lambda q: idx.buscar(q, k=10), k=k)[f"hit@{k}"]
    fila["mrr"] = evaluar(gold, lambda q: idx.buscar(q, k=10), k=5)["mrr"]
    fila["ctx tokens @5"] = round(np.mean([sum(contar(c["texto"]) for c in idx.buscar(q["pregunta"], k=5)) for q in gold if q["evidencia"]]))
    return fila
secc = chunk_seccion_contexto(docs)
idx_secc = Indice(modelo, secc)
filas = [fila_barrido("sección", "-", sec, idx_sec), fila_barrido("sección+ctx", "-", secc, idx_secc)]
for tam in (200, 500, 1000):
    ch = chunk_fijo(docs, tam, int(tam * 0.10), contar)
    filas.append(fila_barrido("fijo (10 % overlap)", tam, ch, Indice(modelo, ch)))
barrido = pd.DataFrame(filas); barrido

**Pregunta 2.** Compara la tabla con tu predicción. Explica en dos líneas el resultado que más te sorprendió con uno de estos tres conceptos: longitud máxima del modelo de embeddings, dilución del embedding en textos largos, granularidad de la métrica.

In [ ]:
# TODO
RESPUESTA_2 = None

## 3. La métrica decide al ganador

Un chunk fijo cuenta como acierto si toca *cualquier* sección de la evidencia (lectura laxa). Con la lectura estricta solo cuenta la sección donde empieza.

In [ ]:
f200 = chunk_fijo(docs, 200, 20, contar)
estricto = [{**c, "secciones": []} for c in f200]
for nombre, ch in (("fijo 200, laxa", f200), ("fijo 200, estricta", estricto), ("sección+ctx", secc)):
    idx = Indice(modelo, ch); r = evaluar(gold, lambda q: idx.buscar(q, k=10), k=5)
    print(f"{nombre:22s} hit@5={r['hit@5']:.2f}  mrr={r['mrr']:.2f}")

**Pregunta 3.** ¿Cuál de las dos lecturas usarías para un asistente que debe citar la sección exacta de la política? ¿Y para uno que solo redacta una respuesta? Una línea cada una.

In [ ]:
# TODO
RESPUESTA_3 = None

## 4. Cambiar el modelo de embeddings

**Predicción:** ¿cuántos puntos de hit@5 pierde un modelo entrenado en inglés sobre este corpus en español? ¿Cero, pocos, muchos?

In [ ]:
# TODO: tu predicción antes de ejecutar.
PREDICCION_MODELO = None
m_en = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
comp = []
for nombre, m, msl in (("multilingüe", modelo, 512), ("inglés", m_en, 256)):
    idx = Indice(m, secc, max_seq_length=msl); r = evaluar(gold, lambda q: idx.buscar(q, k=10), k=5)
    comp.append({"modelo": nombre, "hit@5": r["hit@5"], "mrr": r["mrr"]})
pd.DataFrame(comp)

## 5. Tu propio gold set (actividad 2)

Diez preguntas como las haría un cliente, con el archivo y el encabezado exacto donde está la respuesta. Al menos dos que crucen dos secciones y una sin respuesta en el corpus (evidencia vacía). Formato idéntico al gold set del curso. Dos ejemplos para copiar el formato:

In [ ]:
# TODO: reemplaza los ejemplos por tus diez preguntas. Las que no tienen respuesta llevan evidencia [].
MI_GOLD = [
    {"id": "m01", "pregunta": "Si cancelo hoy, ¿cuánto tiempo puedo seguir consultando mis facturas?", "tipo": "conceptual",
     "evidencia": [{"archivo": "07-politica-cancelacion-y-eliminacion.md", "seccion": "Qué pasa con los datos al cancelar"}]},
    {"id": "m02", "pregunta": "¿Facturio ofrece capacitación presencial?", "tipo": "sin_respuesta", "evidencia": []},
]
# Verifica que cada evidencia exista en el corpus (archivo y encabezado exactos).
encabezados = {a: {s["seccion"] for s in secciones(t)} for a, t in docs.items()}
for q in MI_GOLD:
    for e in q["evidencia"]:
        ok = e["archivo"] in encabezados and e["seccion"] in encabezados[e["archivo"]]
        print(("ok      " if ok else "NO EXISTE"), q["id"], e["archivo"], "/", e["seccion"])

In [ ]:
if MI_GOLD:
    for nombre, ch in (("sección+ctx", secc), ("fijo 200", f200)):
        idx = Indice(modelo, ch); r = evaluar(MI_GOLD, lambda q: idx.buscar(q, k=10), k=5)
        print(f"{nombre:12s} n={r['preguntas']}  hit@5={r['hit@5']:.2f}  mrr={r['mrr']:.2f}  fallan: {[f['id'] for f in r['por_pregunta'] if f['hit'] == 0]}")

**Pregunta 4.** Para cada pregunta tuya que falle en ambas configuraciones: ¿la pregunta era ambigua, la evidencia también estaba en otra sección, o el retriever falló de verdad?

In [ ]:
# TODO: un diagnóstico por pregunta fallida.
DIAGNOSTICO = {}

## 6. Un filtro por metadata

Elige una pregunta del gold set del curso que falle con sección+ctx y prueba un filtro por `tipo` (manual, faq, politica, terminos, catalogo, notas, seguridad, comunicado). Luego responde: en producción, ¿quién decidiría ese filtro?

In [ ]:
g = {x["id"]: x for x in gold}
r = evaluar(gold, lambda q: idx_secc.buscar(q, k=10), k=5)
print("fallan con sección+ctx:", [f["id"] for f in r["por_pregunta"] if f["hit"] == 0])
# TODO: elige una y un tipo.
QID = "q05"
TIPO = "manual"
sin = idx_secc.buscar(g[QID]["pregunta"], k=3); con = idx_secc.buscar(g[QID]["pregunta"], k=3, filtro=lambda c: c["tipo"] == TIPO)
print(g[QID]["pregunta"]); print("evidencia:", [(e["archivo"][:2], e["seccion"]) for e in g[QID]["evidencia"]])
print("sin filtro:", [(c["archivo"][:2], c["seccion"]) for c in sin]); print("con filtro:", [(c["archivo"][:2], c["seccion"]) for c in con])

In [ ]:
# TODO: ¿quién decide el filtro en producción y qué pasa si se equivoca?
RESPUESTA_6 = None

## 7. Construir el contexto: orden y presupuesto

En clase se vio que, con la misma evidencia, la posición del mejor fragmento y el presupuesto de tokens cambian la respuesta. Aquí se repite el experimento con otra pregunta del gold set.

**Predicción 5.** Para `q29`, ¿con qué presupuesto (150, 300, 600 o 900 tokens) esperas la primera respuesta correcta? ¿Y crees que poner el mejor fragmento al final la empeora?

In [ ]:
# TODO: tus predicciones antes de ejecutar.
PRED_PRESUPUESTO_MINIMO = None   # 150, 300, 600 o 900
PRED_POSICION_EMPEORA = None     # True o False

In [ ]:
def contar_tokens(texto: str) -> int:
    """Aproximación estable sin tokenizer: 1 token por cada 3.5 caracteres en español."""
    return int(len(texto) / 3.5) + 1


def construir_contexto(chunks, presupuesto_tokens=900, orden="mejor_primero"):
    """Selecciona chunks sin repetir (archivo, sección) hasta el presupuesto y los numera para citar."""
    elegidos, vistos, usados = [], set(), 0
    for c in chunks:
        clave = (c["archivo"], c["seccion"])
        if clave in vistos:
            continue
        t = contar_tokens(c["texto"])
        if usados + t > presupuesto_tokens:
            continue
        vistos.add(clave); usados += t; elegidos.append(c)
    if orden == "mejor_ultimo":
        elegidos = elegidos[::-1]
    partes = [f"[{i}] ({c['archivo']} · {c.get('ruta') or c['seccion']})\n{c['texto']}" for i, c in enumerate(elegidos, start=1)]
    return "\n\n".join(partes), elegidos


SISTEMA_QA = ("Eres el asistente de soporte de Facturio. Responde en español, en una o dos oraciones, "
              "únicamente con la información de los fragmentos numerados. Cita el fragmento con [n]. "
              "Si los fragmentos no contienen la respuesta, di exactamente: 'No tengo esa información en la documentación.'")


class LLM:
    """Cliente mínimo (copia de shared/llm.py): OpenAI si hay llave, modelo local si no."""
    def __init__(self, model_local="Qwen/Qwen2.5-1.5B-Instruct", model_api="gpt-5-mini"):
        self.backend = "openai" if os.environ.get("OPENAI_API_KEY") else "local"
        if self.backend == "openai":
            from openai import OpenAI
            self.client, self.model = OpenAI(), model_api
        else:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer
            self.tok = AutoTokenizer.from_pretrained(model_local)
            self.net = AutoModelForCausalLM.from_pretrained(model_local, dtype=torch.float32).eval()
            self.model = model_local

    def chat(self, user, system, max_output_tokens=160):
        t0 = time.perf_counter()
        if self.backend == "openai":
            params = dict(model=self.model, instructions=system, input=user, max_output_tokens=max_output_tokens)
            # Reasoning models (gpt-5*, o*): gastan max_output_tokens en razonar; con "minimal" la respuesta
            # cabe en presupuestos pequeños. 
            # Ejercicio: probar: 
            # "none"    Sin razonamiento adicional. Menor latencia.
            # "minimal"    Razonamiento mínimo.
            # "low"    Razonamiento ligero.
            # "medium"    Razonamiento moderado.
            # "high"    Razonamiento profundo.
            # "xhigh"    Razonamiento muy profundo.
            # Summary: la cadena de razonamiento viaja cifrada (encrypted_content) y no se puede leer; se puede
            # pedir un resumen generado por el proveedor con reasoning["summary"]:
            # "auto"    El proveedor elige el nivel de detalle.
            # "concise"    Resumen breve.
            # "detailed"    Resumen paso a paso.
            # Ejemplo: params["reasoning"] = {"effort": "medium", "summary": "auto"}
            # Ver https://developers.openai.com/api/reference/resources/responses/methods/create#responses-create-reasoning
            # https://developers.openai.com/api/docs/models/gpt-5-mini (especifico de este modelo)
            # https://developers.openai.com/api/docs/guides/reasoning?api-mode=responses
            if self.model.startswith(("gpt-5", "o1", "o3", "o4")):
                params["reasoning"] = {"effort": "minimal"}
            r = self.client.responses.create(**params)
            return {"text": r.output_text.strip(), "in": r.usage.input_tokens, "out": r.usage.output_tokens, "s": time.perf_counter() - t0}
        import torch
        texto = self.tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": user}],
                                             add_generation_prompt=True, tokenize=False)
        ids = self.tok(texto, return_tensors="pt").input_ids
        with torch.no_grad():
            out = self.net.generate(ids, max_new_tokens=max_output_tokens, do_sample=False, pad_token_id=self.tok.eos_token_id)
        nuevos = out[0, ids.shape[1]:]
        return {"text": self.tok.decode(nuevos, skip_special_tokens=True).strip(), "in": int(ids.shape[1]), "out": int(nuevos.shape[0]), "s": time.perf_counter() - t0}


def responder(llm, pregunta, retriever, presupuesto=900, orden="mejor_primero", k=8):
    chunks = retriever.buscar(pregunta, k)
    contexto, usados = construir_contexto(chunks, presupuesto, orden)
    r = llm.chat(f"<fragmentos>\n{contexto}\n</fragmentos>\n\n<pregunta>\n{pregunta}\n</pregunta>", SISTEMA_QA)
    r.update({"chunks": usados, "contexto_tokens": contar_tokens(contexto)})
    return r


llm = LLM()
print("backend:", llm.backend, llm.model)
secc = chunk_seccion_contexto(docs)
idx_secc = Indice(modelo, secc)
qid = "q29"
print(g[qid]["pregunta"], "\nesperada:", g[qid]["respuesta_esperada"], "\n")
for presupuesto in [150, 300, 600, 900]:
    for orden in ["mejor_primero", "mejor_ultimo"]:
        r = responder(llm, g[qid]["pregunta"], idx_secc, presupuesto=presupuesto, orden=orden)
        print(f"presupuesto={presupuesto:4d} {orden:14s} fragmentos={len(r['chunks'])} tokens_in={r['in']:5d} -> {r['text'][:100]}")

**Pregunta 5.** ¿Cuál fue el presupuesto mínimo con respuesta correcta? ¿La posición cambió la respuesta? Si fijaras un presupuesto para la mesa de soporte con un millón de consultas al mes, ¿cuál elegirías y cuántos tokens de entrada al mes implica?

In [ ]:
# TODO: responde en texto, con los números de la tabla de arriba.

## 8. Lectura: parsing de PDF y vector stores

Dos temas que salieron de la sesión por tiempo. Corren aquí y conviene leerlos antes de la práctica.

**Parsing.** El corpus del curso es markdown con encabezados reales, y por eso el chunking por sección funciona. Un PDF pierde esa estructura: al extraer el texto, los encabezados se vuelven líneas más y las tablas se aplanan. La celda convierte un documento del corpus a PDF, lo vuelve a leer y cuenta qué sobrevive.

In [ ]:
# Ida y vuelta por PDF: qué pierde el parser
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import tempfile
nombre = "08-terminos-del-servicio-y-sla.md"
texto = docs[nombre]
ruta_pdf = os.path.join(tempfile.gettempdir(), "terminos.pdf")
lineas = texto.splitlines()
with PdfPages(ruta_pdf) as pdf:
    for i in range(0, len(lineas), 55):
        fig = plt.figure(figsize=(8.5, 11)); fig.text(0.05, 0.95, "\n".join(lineas[i:i+55]), va="top", family="monospace", fontsize=7)
        pdf.savefig(fig); plt.close(fig)
try:
    from pypdf import PdfReader
    extraido = "\n".join(p.extract_text() for p in PdfReader(ruta_pdf).pages)
    enc_md = sum(1 for l in lineas if l.startswith("#"))
    enc_pdf = sum(1 for l in extraido.splitlines() if l.startswith("#"))
    print(f"encabezados en markdown: {enc_md} | detectables tras el PDF: {enc_pdf}")
    print(f"tablas en markdown: {sum(1 for l in lineas if l.startswith('|'))} filas | en el PDF se vuelven texto plano")
    print("primeras líneas extraídas:\n", extraido[:400])
except ImportError:
    print("pypdf no instalado: pip install pypdf")

**Vector stores.** El índice de numpy de clase hace exactamente lo que hace una base vectorial: guardar vectores y buscar por similitud. FAISS lo acelera para millones de vectores; Chroma agrega ids, metadata, filtros y persistencia. Ninguno cambia los resultados con 200 chunks: cambia lo que la aplicación tiene que escribir.

In [ ]:
E = idx_secc.E if hasattr(idx_secc, "E") else modelo.encode([c["texto"] for c in secc], normalize_embeddings=True)
import numpy as np
E = np.asarray(E, dtype="float32")
q = modelo.encode([g["q05"]["pregunta"]], normalize_embeddings=True).astype("float32")
try:
    import faiss
    ix = faiss.IndexFlatIP(E.shape[1]); ix.add(E)
    D, I = ix.search(q, 3)
    print("FAISS top-3:", [(secc[i]["archivo"], secc[i]["seccion"], round(float(d), 3)) for d, i in zip(D[0], I[0])])
except ImportError:
    print("faiss no instalado: pip install faiss-cpu")
try:
    import chromadb
    col = chromadb.Client().get_or_create_collection("facturio", metadata={"hnsw:space": "cosine"})
    col.add(ids=[f"c{i}" for i in range(len(secc))], embeddings=E.tolist(),
            metadatas=[{"archivo": c["archivo"], "seccion": c["seccion"], "tipo": c.get("tipo", "")} for c in secc],
            documents=[c["texto"] for c in secc])
    res = col.query(query_embeddings=q.tolist(), n_results=3, where={"tipo": "manual"})
    print("Chroma top-3 con filtro tipo=manual:", [(m["archivo"], m["seccion"]) for m in res["metadatas"][0]])
except ImportError:
    print("chromadb no instalado: pip install chromadb")

## 9. Cierre

Con lo medido hoy, escribe la configuración que recomendarías para la mesa de soporte de Facturio (chunking, modelo, k, filtros) y una línea con la falla que sigue sin resolverse.

In [ ]:
# TODO
MI_CONFIGURACION = {"chunking": None, "modelo": None, "k": None, "filtros": None, "lo que sigue fallando": None}
print(json.dumps({"prediccion": MI_PREDICCION, "configuracion": MI_CONFIGURACION}, ensure_ascii=False, indent=1))

### Para la práctica 6

`practice/practice.md`: el barrido completo con predicciones registradas, dos modelos de embeddings, diez pares gold propios, un caso de filtering y la recomendación justificada con la tabla. `practice-starter.ipynb` trae las celdas listas.